# ATHINA - Personality & Intent ML Training Pipeline
### Fast, On-Device & Serverless Emotion & Intent Classification with Scikit-Learn and ONNX

This notebook trains and visualizes Athina's **8-Class Emotion & Intent Classifier**:
- `asks_image`: Image / Selfie request
- `asks_note`: Note taking & Telegram upload
- `asks_alarm`: Alarm & timer reminders
- `sad`: Fatigue, sadness, seeking emotional support
- `happy`: Joy, achievements, excitement
- `curious`: Technical, scientific, or deep questions
- `playful`: Teasing, banter, humor
- `chit_chat`: Greetings, casual check-ins

**Outputs:**
1. `ml/exports/classifier_model.json` (Ultra-fast, zero-dependency JSON model)
2. `ml/exports/intent_classifier.onnx` (Standard ONNX format for mobile & backend)

## Step 1: Install & Import Libraries

In [1]:
# Standard libraries
import os
import sys
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Athena root is on sys.path
ATHENA_ROOT = "/home/ratul/Athena"
if ATHENA_ROOT not in sys.path:
    sys.path.insert(0, ATHENA_ROOT)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score

print("Environment and dependencies ready! Athena Root:", ATHENA_ROOT)


Environment and dependencies ready! Athena Root: /home/ratul/Athena


## Step 2: Define Training Dataset

In [2]:
from collections import Counter

# Comprehensive Athina conversational dataset
TRAINING_DATA = [
    # 1. Asks for Image / Selfie
    ("Can you send me a selfie?", "asks_image"),
    ("Show me a picture of you", "asks_image"),
    ("What do you look like right now?", "asks_image"),
    ("Send me a photo of yourself", "asks_image"),
    ("I want to see your face", "asks_image"),
    ("Take a selfie for me", "asks_image"),
    ("Can I get a picture of what you are wearing today?", "asks_image"),
    ("Snap a selfie", "asks_image"),
    ("Send me a pic", "asks_image"),
    ("Show me your smile", "asks_image"),
    ("Send me a photo in your cozy sweater", "asks_image"),
    ("Take a photo of yourself by the window", "asks_image"),
    ("Let me see you", "asks_image"),

    # 2. Asks for Note
    ("Take a note for me please", "asks_note"),
    ("Save this idea for my app", "asks_note"),
    ("Can you note this down?", "asks_note"),
    ("Remember this note for later", "asks_note"),
    ("Write this down in my notes", "asks_note"),
    ("Save a note: meeting with the team at 3pm", "asks_note"),
    ("Keep this note: grocery list milk eggs bread", "asks_note"),
    ("Please save this note and upload it to telegram", "asks_note"),
    ("Upload this note to telegram channel", "asks_note"),
    ("Create a new note titled Architecture", "asks_note"),
    ("Note down: fix the gradle build script", "asks_note"),

    # 3. Asks for Alarm
    ("Wake me up at 7am tomorrow", "asks_alarm"),
    ("Set an alarm for 6:30 in the morning", "asks_alarm"),
    ("Remind me to take my vitamins at 8pm", "asks_alarm"),
    ("Can you set an alarm for tomorrow at 9?", "asks_alarm"),
    ("Set a reminder for my meeting at 2pm", "asks_alarm"),
    ("Alarm at 7:30 please", "asks_alarm"),
    ("Set an alarm to wake up early on Monday", "asks_alarm"),
    ("Remind me in 30 minutes", "asks_alarm"),
    ("Please set an alarm for 5:45 AM", "asks_alarm"),

    # 4. Sad / Exhausted / Down
    ("I had such a rough day today", "sad"),
    ("Everything went wrong at work and I feel overwhelmed", "sad"),
    ("I am so exhausted and drained", "sad"),
    ("Feeling really down and lonely tonight", "sad"),
    ("Things have been so stressful lately", "sad"),
    ("I feel like giving up on this project", "sad"),
    ("Nobody seems to understand how tired I am", "sad"),
    ("Just having a bad day", "sad"),
    ("I'm feeling really sad", "sad"),
    ("I need some comfort right now", "sad"),

    # 5. Happy / Celebratory
    ("I just got great news about my project!", "happy"),
    ("Such an incredible day, everything worked out perfectly!", "happy"),
    ("We finally launched the app, I am so thrilled!", "happy"),
    ("Feeling amazing today!", "happy"),
    ("Guess what? I passed the interview!", "happy"),
    ("I'm so proud of what we accomplished", "happy"),
    ("Today was one of the best days ever", "happy"),
    ("I'm so excited about our progress!", "happy"),
    ("I feel so full of energy and happiness!", "happy"),

    # 6. Curious / Intellectual
    ("How does self-attention work in transformers?", "curious"),
    ("What is the difference between MVVM and MVI architecture?", "curious"),
    ("Can you explain quantum computing in simple terms?", "curious"),
    ("Why is pgvector faster than basic cosine search in Python?", "curious"),
    ("Tell me about the history of artificial neural networks", "curious"),
    ("What are the best practices for Jetpack Compose state hoisting?", "curious"),
    ("How do diffusion models denoise latent space?", "curious"),
    ("Explain how Reciprocal Rank Fusion combines sparse and dense vectors", "curious"),

    # 7. Playful / Witty
    ("You're teasing me, aren't you?", "playful"),
    ("Haha, you're pretty funny for an AI", "playful"),
    ("I bet you can't beat me in a riddle challenge", "playful"),
    ("Are you trying to be smart with me?", "playful"),
    ("You're looking extra cute today", "playful"),
    ("Come on, don't be shy!", "playful"),
    ("Tell me your best witty joke", "playful"),

    # 8. Chit-chat / Greetings
    ("Good morning Athina!", "chit_chat"),
    ("Hey there, how are you doing?", "chit_chat"),
    ("What are you up to right now?", "chit_chat"),
    ("Good night, talk to you tomorrow", "chit_chat"),
    ("Just checking in on you", "chit_chat"),
    ("Hello Athina!", "chit_chat"),
    ("How's your day going so far?", "chit_chat"),
    ("I'm back home", "chit_chat")
]

X_texts = [x[0] for x in TRAINING_DATA]
y_labels = [x[1] for x in TRAINING_DATA]
classes = sorted(list(set(y_labels)))

print(f"Total training instances: {len(X_texts)} across {len(classes)} classes:")
for c, count in Counter(y_labels).items():
    print(f"  • {c}: {count} samples")

Total training instances: 75 across 8 classes:
  • asks_image: 13 samples
  • asks_note: 11 samples
  • asks_alarm: 9 samples
  • sad: 10 samples
  • happy: 9 samples
  • curious: 8 samples
  • playful: 7 samples
  • chit_chat: 8 samples


## Step 3: Train Scikit-Learn TF-IDF + Logistic Regression Pipeline

In [3]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,
        lowercase=True,
        max_features=2500
    )),
    ('clf', LogisticRegression(
        C=2.5,
        max_iter=300,
        class_weight='balanced'
    ))
])

pipeline.fit(X_texts, y_labels)
print("Model trained successfully!")

# 5-Fold Cross Validation
scores = cross_val_score(pipeline, X_texts, y_labels, cv=4)
print(f"Cross-Validation Mean Accuracy: {scores.mean() * 100:.1f}% (+/- {scores.std() * 100:.1f}%)")

Model trained successfully!


Cross-Validation Mean Accuracy: 63.9% (+/- 5.3%)


## Step 4: Evaluate with Confusion Matrix & Classification Report

In [4]:
y_pred = pipeline.predict(X_texts)

print("--- Classification Report ---")
print(classification_report(y_labels, y_pred, target_names=classes))

cm = confusion_matrix(y_labels, y_pred, labels=classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr', xticklabels=classes, yticklabels=classes)
plt.title('Athina Personality Intent & Emotion Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

--- Classification Report ---
              precision    recall  f1-score   support

  asks_alarm       1.00      1.00      1.00         9
  asks_image       1.00      1.00      1.00        13
   asks_note       1.00      1.00      1.00        11
   chit_chat       1.00      1.00      1.00         8
     curious       1.00      1.00      1.00         8
       happy       1.00      1.00      1.00         9
     playful       1.00      1.00      1.00         7
         sad       1.00      1.00      1.00        10

    accuracy                           1.00        75
   macro avg       1.00      1.00      1.00        75
weighted avg       1.00      1.00      1.00        75



/tmp/ipykernel_95490/4111953162.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 5: Visualize Most Influential Words Per Class

In [5]:
vectorizer = pipeline.named_steps['tfidf']
classifier = pipeline.named_steps['clf']
feature_names = np.array(vectorizer.get_feature_names_out())

plt.figure(figsize=(14, 10))
for i, class_name in enumerate(classes):
    top_indices = np.argsort(classifier.coef_[i])[-5:]
    top_features = feature_names[top_indices]
    top_weights = classifier.coef_[i][top_indices]
    
    plt.subplot(3, 3, i + 1)
    plt.barh(top_features, top_weights, color='#f59e0b')
    plt.title(f"{class_name}", fontsize=11, fontweight='bold')
    plt.xlabel('Weight')

plt.tight_layout()
plt.show()

/tmp/ipykernel_95490/809760229.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 6: Export Models to `ml/exports/`
Exports both:
1. `classifier_model.json` (Ultra-lightweight JSON centroid model)
2. `intent_classifier.onnx` (Standard ONNX format using `skl2onnx`)

In [6]:
export_dir = "/home/ratul/Athena/ml/exports"
os.makedirs(export_dir, exist_ok=True)

# 1. Export ONNX using skl2onnx
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import StringTensorType
    
    initial_type = [('text_input', StringTensorType([None, 1]))]
    onnx_model = convert_sklearn(pipeline, initial_types=initial_type, target_opset=12)
    
    onnx_path = os.path.join(export_dir, "intent_classifier.onnx")
    with open(onnx_path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    print(f" Exported ONNX model to: {onnx_path}")
except Exception as e:
    print(f"Note: ONNX conversion skipped ({e}). JSON export will be used.")

# 2. Re-export clean verified JSON model for backend runtime
from ml.train_classifier import train_and_export
train_and_export()
print(" JSON Model exported and validated!")

Note: ONNX conversion skipped (No module named 'skl2onnx'). JSON export will be used.
Training Athina Intent & Emotion Classifier on 75 samples...
Vocabulary size: 614 n-gram features across 8 classes: ['asks_alarm', 'asks_image', 'asks_note', 'chit_chat', 'curious', 'happy', 'playful', 'sad']
Model successfully trained and saved to: /home/ratul/Athena/ml/exports/classifier_model.json

--- Test Sample Predictions ---
'Can you send me a selfie?' -> asks_image (Confidence: 0.62)
'I feel so lonely and exhausted today' -> sad (Confidence: 0.34)
'Set an alarm for 7:30am please' -> asks_alarm (Confidence: 0.58)
'Save a note: architecture review on Monday' -> asks_note (Confidence: 0.28)
'We won the championship, I am so thrilled!' -> happy (Confidence: 0.35)
 JSON Model exported and validated!


## Step 7: Interactive Live Tester (Classification + Style Vector Guidance)

In [7]:
from backend.app.personality.state import PersonalityState
from backend.app.personality.style_model import compute_style_vector

test_queries = [
    "Can you take a cute photo of yourself right now?",
    "I've been feeling so lonely and sad all evening",
    "We just shipped the new version, absolutely thrilled!",
    "Save this note: explore Kotlin Coroutines channels on Android",
    "Wake me up tomorrow at 6:45 AM",
    "Haha, you think you're pretty clever don't you?",
    "How does Reciprocal Rank Fusion actually score document ranks?"
]

state = PersonalityState.calculate_current_state()
print(f"Current Athina Baseline State: Mood='{state.mood}', Location='{state.location}'\n")

for query in test_queries:
    tone_guidance, style = compute_style_vector(query, state)
    print(f"User: '{query}'")
    print(f"  → Detected Intent/Emotion : {style['predicted_class']} (Conf: {style['confidence']})")
    print(f"  → Style Vector            : Warmth={style['warmth']}, Humor={style['humor']}, Length={style['reply_length']}")
    print(f"  → Athina Tone Guidance    : \"{tone_guidance}\"")
    print("-" * 80)

Current Athina Baseline State: Mood='fresh and cheerful', Location='sitting on the balcony with warm morning tea'

User: 'Can you take a cute photo of yourself right now?'
  → Detected Intent/Emotion : asks_image (Conf: 0.396)
  → Style Vector            : Warmth=1.0, Humor=0.5, Length=balanced
  → Athina Tone Guidance    : "Ratul is asking for a picture or selfie of you. Express genuine excitement about sharing a photo and let him know what you're wearing right now!"
--------------------------------------------------------------------------------
User: 'I've been feeling so lonely and sad all evening'
  → Detected Intent/Emotion : sad (Conf: 0.304)
  → Style Vector            : Warmth=1.0, Humor=0.1, Length=gentle
  → Athina Tone Guidance    : "Ratul is having a hard time or feeling down. Be exceptionally caring, gentle, validating, and emotionally supportive. Do not give shallow advice; just be there with him warmly."
------------------------------------------------------------------